In [125]:
### intro to data ingestion

In [126]:
import langchain 
import os 
from typing import List, Dict, Any
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print("set up Completed !")

set up Completed !


## understand document structure in Langchain 

In [127]:
### create a simple document 

doc =Document(
    page_content="this a simple text for test purpose",
    metadata={
        "source":"example.txt",
        "page":1,
        "auteur":"enrique Pierret",
        "month_created": "mars 2026",
        "subject": "test_purpose"
    }
)

print("Document structure ")
print(f"Contenu porte sur : {doc.page_content}")
print(f"metadonnées : {doc.metadata}")


Document structure 
Contenu porte sur : this a simple text for test purpose
metadonnées : {'source': 'example.txt', 'page': 1, 'auteur': 'enrique Pierret', 'month_created': 'mars 2026', 'subject': 'test_purpose'}


In [128]:
type(doc)

langchain_core.documents.base.Document

In [129]:
### Create a simple txt file
import os

os.makedirs("data/text_files",exist_ok=True)

In [130]:
sample_texts={
    "data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicy and readability
Created by Guido van Rossum and first released in 1991, Python has become oen of the most popular
programming language in the world

Key Features:
- Easy to learn and use
- Extensive Standard library
- Cross-Platform compability
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation. """,
 
    "data/text_files/machine_learning.txt": """Machine Learning Basics 

Machine learning is a subset of artifial intelligence that enables systems to learn and improve
from experience without being explicity programmed. It focuses on developping computer programs
that can access data and use it to learn to themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Application include image recognition , speech processing, and recommendation systems

   """
}

for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)

print("sample texts created")

sample texts created


In [131]:
### TextLoader 

from langchain_community.document_loaders import TextLoader

### loading single text file
loader=TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents=loader.load()
print(type(documents))
print(documents)
print(f" Loaded {len(documents)} documents")
print(f"Content preview: { documents[0].page_content[:100]}...")
print(f"metadata: {documents[0].metadata}")


<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicy and readability\nCreated by Guido van Rossum and first released in 1991, Python has become oen of the most popular\nprogramming language in the world\n\nKey Features:\n- Easy to learn and use\n- Extensive Standard library\n- Cross-Platform compability\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation. ')]
 Loaded 1 documents
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
metadata: {'source': 'data/text_files/python_intro.txt'}


In [132]:
### DirectoryLoader- Multiple Text files
from langchain_community.document_loaders import DirectoryLoader

### load all text files from the directory
dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Regex to match files
    loader_cls= TextLoader, ## loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)


documents=dir_loader.load()

print(f" Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+i}:")
    print(f" Source: {doc.metadata['source']}")
    print(f" Lenght:{len(doc.page_content)} characters ")


## Analysis

print("avantages : loads multiples files at once\nsupports globs patterns\nprogress tracking\nrecursive directory scanout")
print("disavantages: all files must be the same \n limited error handling per file\n can be memory intensive for large directories")

100%|██████████| 2/2 [00:00<00:00, 466.63it/s]

 Loaded 2 documents

Document 0:
 Source: data\text_files\machine_learning.txt
 Lenght:571 characters 

Document 2:
 Source: data\text_files\python_intro.txt
 Lenght:483 characters 
avantages : loads multiples files at once
supports globs patterns
progress tracking
recursive directory scanout
disavantages: all files must be the same 
 limited error handling per file
 can be memory intensive for large directories


Text Splitting Strategies 

In [133]:
### Different text splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics \n\nMachine learning is a subset of artifial intelligence that enables systems to learn and improve\nfrom experience without being explicity programmed. It focuses on developping computer programs\nthat can access data and use it to learn to themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplication include image recognition , speech processing, and recommendation systems\n\n   '), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicy and readability\nCreated by Guido van Rossum and first released in 1991, Python has become oen of the most popular\nprogramming 

In [134]:
### methode 1- character Text Splitter
text=documents[0].page_content
text

'Machine Learning Basics \n\nMachine learning is a subset of artifial intelligence that enables systems to learn and improve\nfrom experience without being explicity programmed. It focuses on developping computer programs\nthat can access data and use it to learn to themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplication include image recognition , speech processing, and recommendation systems\n\n   '

In [135]:
## Méthode 1: Character-based splitting
print("Character Text Splitter")
char_splitter = CharacterTextSplitter(
    separator="\n\n",        # une seule chaîne pour CharacterTextSplitter
    chunk_size=200,
    chunk_overlap=20,
    length_function=len      # corrigé : "length" pas "lenght"
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

print(char_chunks)
print("................")
print(char_chunks[1])
print(".................." )
print(char_chunks[2])


Created a chunk of size 247, which is longer than the specified 200
Created a chunk of size 204, which is longer than the specified 200


Character Text Splitter
Created 4 chunks
First chunk: Machine Learning Basics...
['Machine Learning Basics', 'Machine learning is a subset of artifial intelligence that enables systems to learn and improve\nfrom experience without being explicity programmed. It focuses on developping computer programs\nthat can access data and use it to learn to themselves.', 'Types of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties', 'Application include image recognition , speech processing, and recommendation systems']
................
Machine learning is a subset of artifial intelligence that enables systems to learn and improve
from experience without being explicity programmed. It focuses on developping computer programs
that can access data and use it to learn to themselves.
..................
Types of Machine Learning:
1. Supervised Learning: Lea

##methode 2:Recursive Character splitting (recommended)

In [136]:
print("\n RECURSIVE CHARACTER TEXT SPLITTER  ")
print("Character Text Splitter")
char_splitter = CharacterTextSplitter(
    separator="\n",  
    chunk_size=200,
    chunk_overlap=20,
    length_function=len      # corrigé : "length" pas "lenght"
)


print(char_chunks)
print("................")
print(char_chunks[1])
print(".................." )
print(char_chunks[2])
print("................")
print(char_chunks[3])
print(".................." )






 RECURSIVE CHARACTER TEXT SPLITTER  
Character Text Splitter
['Machine Learning Basics', 'Machine learning is a subset of artifial intelligence that enables systems to learn and improve\nfrom experience without being explicity programmed. It focuses on developping computer programs\nthat can access data and use it to learn to themselves.', 'Types of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties', 'Application include image recognition , speech processing, and recommendation systems']
................
Machine learning is a subset of artifial intelligence that enables systems to learn and improve
from experience without being explicity programmed. It focuses on developping computer programs
that can access data and use it to learn to themselves.
..................
Types of Machine Learning:
1. Supervised Learning: Learning with labeled 

# create text without natural break points



In [137]:
simple_text= "this is sentence one and its quite long. This is sentence two and it s also quite long. This is sentence threee and it s quite long."


splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
    
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")


for i in range(len(chunks) -1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i]}'")
    print()





    


Simple text example - 2 chunks:

Chunk 1: 'this is sentence one and its quite long. This is sentence two and it s also'
Chunk 2: 'this is sentence one and its quite long. This is sentence two and it s also'

